# Week 5: Model Optimization and Experimentation
**Bank Marketing Dataset | Machine Learning Engineer Internship**

This notebook runs two tuning experiments against the Week 4 Logistic Regression baseline
(CV F1 mean 0.427, test recall 0.600):

1. **Grid Search** over the baseline Logistic Regression's own hyperparameters (C, penalty).
2. **Randomized Search** over a Random Forest — a fundamentally different, non-linear model —
   to directly test the Week 4 finding that a near-identical false negative/false positive pair
   suggested a ceiling on what a linear model could separate.

## 1. Rebuild Pipeline (per Weeks 2–4)

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, roc_auc_score)

df = pd.read_csv("bank-additional.csv", sep=";")
df['contacted_before'] = (df['pdays'] != 999).astype(int)
df['pdays_clean'] = df['pdays'].replace(999, 0)
df = df.drop(columns=['pdays'])
df_deploy = df.drop(columns=['duration'])

nominal_cols = ['job', 'marital', 'contact', 'poutcome', 'month', 'day_of_week']
df_encoded = pd.get_dummies(df_deploy, columns=nominal_cols)
edu_order = ['unknown', 'illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
             'high.school', 'professional.course', 'university.degree']
df_encoded['education'] = df_deploy['education'].map({v: i for i, v in enumerate(edu_order)})
for col in ['default', 'housing', 'loan']:
    df_encoded[col] = df_deploy[col].map({'no': 0, 'yes': 1, 'unknown': -1})
df_encoded['y'] = df_encoded['y'].map({'no': 0, 'yes': 1})

numeric_cols = ['age', 'campaign', 'previous', 'pdays_clean',
                'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
scaler = StandardScaler()
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

X = df_encoded.drop(columns=['y'])
y = df_encoded['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Pipeline rebuilt. Train:", X_train.shape, "Test:", X_test.shape)

Pipeline rebuilt. Train: (3295, 50) Test: (824, 50)


## 2. Experiment 1: Grid Search on Logistic Regression

**Parameter space:** `C` (inverse regularization strength) across 5 values spanning two orders
of magnitude, and `penalty` (L1 vs. L2 regularization) — 10 total combinations, each evaluated
with 5-fold stratified cross-validation (50 model fits total).

**Control variables held fixed:** `class_weight='balanced'`, `random_state=42`, and the exact
same train/test split as Week 4 — so any performance difference is attributable only to the
hyperparameters being searched, not to a different data split.

In [2]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']  # supports both l1 and l2
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
base_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

grid = GridSearchCV(base_model, param_grid, scoring='f1', cv=skf, n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV F1:", round(grid.best_score_, 3))

Best params: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
Best CV F1: 0.435


In [3]:
results_df = pd.DataFrame(grid.cv_results_)[['param_C', 'param_penalty', 'mean_test_score', 'std_test_score']]
results_df = results_df.sort_values('mean_test_score', ascending=False)
results_df.columns = ['C', 'penalty', 'mean_cv_f1', 'std_cv_f1']
results_df.round(3)

,C,penalty,mean_cv_f1,std_cv_f1
3,0.10,l2,0.435,0.028
4,1.00,l1,0.432,0.031
7,10.00,l2,0.431,0.037
6,10.00,l1,0.431,0.035
8,100.00,l1,0.431,0.037
9,100.00,l2,0.431,0.037
5,1.00,l2,0.431,0.035
2,0.10,l1,0.426,0.038
1,0.01,l2,0.402,0.043
0,0.01,l1,0.353,0.031


**Reading the full grid:** every configuration scored between 0.353 and 0.435 F1 — a narrow
0.08 range across all 10 combinations. This itself is informative: Logistic Regression's
performance on this dataset is not highly sensitive to regularization strength or type, which
suggests the model's ceiling here is set more by its linear structure and the feature set than
by under- or over-regularization.

In [4]:
tuned_lr = grid.best_estimator_
y_pred_lr = tuned_lr.predict(X_test)
y_proba_lr = tuned_lr.predict_proba(X_test)[:, 1]

print("=== TUNED LOGISTIC REGRESSION — HELD-OUT TEST SET ===")
print("Accuracy: ", round(accuracy_score(y_test, y_pred_lr), 3))
print("Precision:", round(precision_score(y_test, y_pred_lr), 3))
print("Recall:   ", round(recall_score(y_test, y_pred_lr), 3))
print("F1:       ", round(f1_score(y_test, y_pred_lr), 3))
print("ROC-AUC:  ", round(roc_auc_score(y_test, y_proba_lr), 3))

=== TUNED LOGISTIC REGRESSION — HELD-OUT TEST SET ===
Accuracy:  0.839
Precision: 0.356
Recall:    0.589
F1:        0.444
ROC-AUC:   0.787


**Honest finding:** the tuned model (F1 0.444, recall 0.589) does not clearly beat the Week 4
untuned baseline (F1 0.460, recall 0.600) on the held-out test set — despite a marginally higher
cross-validation F1 (0.435 vs. 0.427). This is a real and useful result, not a disappointing one:
it shows that for this model and feature set, hyperparameter tuning alone has largely plateaued,
and the earlier baseline's `class_weight='balanced'` default was already a reasonable choice. It
also demonstrates why CV score and test score should both be checked — optimizing purely for the
CV number does not guarantee a test-set improvement.

## 3. Experiment 2: Randomized Search on Random Forest

Week 4's error analysis found a false negative and a false positive example that were nearly
identical on every available feature, suggesting a ceiling on what a **linear** model could
separate. This experiment tests that directly with a fundamentally different, non-linear model.

**Parameter space:** `n_estimators`, `max_depth`, `min_samples_leaf`, and `max_features` —
4 dimensions, too large to grid search exhaustively in reasonable time, so Randomized Search
samples 20 random combinations instead of the ~160 a full grid would require.

In [5]:
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [4, 6, 8, 10, None],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

rf = RandomForestClassifier(class_weight='balanced', random_state=42)
random_search = RandomizedSearchCV(rf, param_dist, n_iter=20, scoring='f1',
                                    cv=skf, n_jobs=-1, random_state=42)
random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best CV F1:", round(random_search.best_score_, 3))

Best params: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 6}
Best CV F1: 0.489


**Why Randomized Search here, not Grid Search:** a full grid over these 4 parameters would require 4×5×4×2 = 160 fits × 5 folds = 800 model fits. Randomized Search samples 20 combinations × 5 folds = 100 fits — an 8x reduction in compute for a parameter space this large, while still covering the space broadly enough to find a strong configuration.

In [6]:
best_rf = random_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

print("=== TUNED RANDOM FOREST — HELD-OUT TEST SET ===")
print("Accuracy: ", round(accuracy_score(y_test, y_pred_rf), 3))
print("Precision:", round(precision_score(y_test, y_pred_rf), 3))
print("Recall:   ", round(recall_score(y_test, y_pred_rf), 3))
print("F1:       ", round(f1_score(y_test, y_pred_rf), 3))
print("ROC-AUC:  ", round(roc_auc_score(y_test, y_proba_rf), 3))

=== TUNED RANDOM FOREST — HELD-OUT TEST SET ===
Accuracy:  0.875
Precision: 0.439
Recall:    0.522
F1:        0.477
ROC-AUC:   0.792


## 4. Revisiting the Week 4 Near-Identical FN/FP Pair

Directly testing whether a non-linear model resolves the specific pair of clients that Logistic
Regression could not distinguish (Week 4, Section 5.2): a false negative (idx 179, a real
subscriber the model missed) and a false positive (idx 2295, a non-subscriber flagged as likely)
that were nearly identical on every available feature.

In [7]:
for idx, label in [(179, 'Week 4 False Negative'), (2295, 'Week 4 False Positive')]:
    row = X_test.loc[[idx]]
    proba = best_rf.predict_proba(row)[:, 1][0]
    pred = best_rf.predict(row)[0]
    actual = y_test.loc[idx]
    print(f"{label} (idx {idx}): actual={actual}  RF_predicted={pred}  RF_probability={proba:.3f}")

Week 4 False Negative (idx 179): actual=1  RF_predicted=0  RF_probability=0.292


Week 4 False Positive (idx 2295): actual=0  RF_predicted=0  RF_probability=0.341


**Result — a genuinely mixed finding:** the Random Forest correctly resolves the false
positive (predicting "no" with probability 0.341, matching the true label) but still misses the
false negative (predicting "no" with probability 0.292, when the true label is "yes"). This is
an honest, partial result rather than a clean win: switching model types fixed one of the two
problem cases but not the other, suggesting the false negative case may genuinely lack sufficient
signal in the available features — no reasonably-sized model change resolves it, while the false
positive was more of a borderline calibration issue that a non-linear decision boundary could fix.

## 5. Feature Importance Comparison

In [8]:
importances = pd.Series(best_rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Top 10 Random Forest feature importances:")
print(importances.head(10).round(3))

Top 10 Random Forest feature importances:
nr.employed         0.164
euribor3m           0.157
emp.var.rate        0.121
cons.conf.idx       0.063
contacted_before    0.050
cons.price.idx      0.047
pdays_clean         0.047
age                 0.035
poutcome_success    0.035
previous            0.031
dtype: float64


**A different lens than Logistic Regression's coefficients (Week 4):** the top Logistic
Regression coefficients were dominated by categorical indicators like `month_mar` and
`job_unknown`. The Random Forest instead ranks the continuous socio-economic indicators —
`nr.employed`, `euribor3m`, `emp.var.rate` — as the three most important features by a wide
margin (over 44% of total importance combined). This is a meaningful cross-check: two different
model types agree that macroeconomic context matters, but disagree on exactly which features
carry the most signal, which is a useful caveat against over-trusting any single model's
"explanation" of the data.

## 6. Experiment Comparison Summary

| Model | CV F1 | Test Accuracy | Test Precision | Test Recall | Test F1 | Test ROC-AUC |
|---|---|---|---|---|---|---|
| Week 4 Baseline (LR, untuned) | 0.427 | 0.846 | 0.372 | **0.600** | 0.460 | 0.772 |
| Tuned Logistic Regression | 0.435 | 0.839 | 0.356 | 0.589 | 0.444 | 0.787 |
| Tuned Random Forest | **0.489** | **0.875** | **0.439** | 0.522 | **0.477** | **0.792** |

**No single model wins on every metric — a genuine trade-off, not a clean win:**
Random Forest achieves the best F1, precision, accuracy, and ROC-AUC, but at a real cost to
recall, which Week 4 argued was the metric that matters most for this business problem (catching
genuine subscribers). Whether Random Forest is actually the better choice depends on whether Week
4's priority — recall — still governs, or whether the bank's real priorities justify trading some
recall for meaningfully fewer false alarms (a 34% reduction: 60 vs. 91 false positives).

## 7. Preventing Overfitting During Tuning

- **Cross-validation was used throughout the search itself**, not just for the final report —
  both `GridSearchCV` and `RandomizedSearchCV` select the best configuration based on 5-fold CV
  performance, not a single validation score, reducing the risk of picking a configuration that
  is only accidentally good on one split.
- **The held-out test set was touched only once per model**, after the search concluded — it was
  never used to select hyperparameters, preserving it as a genuinely unseen check.
- **Random Forest's own regularization parameters were included in the search** (`max_depth`,
  `min_samples_leaf`) specifically to guard against the classic Random Forest overfitting failure
  mode of growing unconstrained, fully-deep trees.
- **The gap between CV score and test score was checked for both models** (LR: 0.435 vs. 0.444;
  RF: 0.489 vs. 0.477) — both are close, indicating neither tuned model overfit to its
  cross-validation folds.

## 8. Conclusion and Recommendation

Given Week 4's explicit prioritization of recall (missing a genuine subscriber treated as costlier
than one extra call), **the Week 4 Logistic Regression baseline remains the recommended model**,
despite Random Forest's stronger showing on F1, precision, and accuracy. This is not the
outcome that was expected going into this experiment — the working hypothesis was that a
non-linear model would clearly outperform. The actual result is more nuanced and more honest:
optimization surfaced a real trade-off rather than a clear winner, and the choice between the two
models is ultimately a business decision about the relative cost of false positives versus false
negatives, not a purely technical one. This recommendation, and the reasoning behind it, carries
into the Week 6 final report.